# Private Documents Using RAG, LangChain, and LLMs

### Setup

In [1]:
%%capture
%pip install -U \
langchain \
langchain-core \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
langchain-chroma \
langchain-classic \
langchain-ibm \
ibm-watsonx-ai \
chromadb \
sentence-transformers \
transformers \
huggingface-hub \
wget

In [2]:
%pip install --upgrade \
numpy \
pandas \
scipy \
scikit-learn

   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   ----------- ---------------------------- 3.7/12.7 MB 25.9 MB/s eta 0:00:01
   ------------------------------ --------- 9.7/12.7 MB 26.5 MB/s eta 0:00:01
   ---------------------------------------- 12.7/12.7 MB 25.1 MB/s  0:00:00
   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
   ------------- -------------------------- 3.7/10.9 MB 20.4 MB/s eta 0:00:01
   ---------------------- ----------------- 6.0/10.9 MB 15.1 MB/s eta 0:00:01
   --------------------------- ------------ 7.6/10.9 MB 12.7 MB/s eta 0:00:01
   --------------------------- ------------ 7.6/10.9 MB 12.7 MB/s eta 0:00:01
   ----------------------------------- ---- 9.7/10.9 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 10.9/10.9 MB 9.0 MB/s  0:00:01

  Attempting uninstall: numpy

    Found existing installation: numpy 2.4.0

   ---------------------------------------- 0/2 [numpy]
    Uninstalling numpy-2.4.0:


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [29]:
%pip install -U python-dotenv
%pip install -U langchain-openai


Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 16.6 MB/s  0:00:00
   ---------------------------------------- 0.0/988.0 kB ? eta -:--:--
   ---------------------------------------- 988.0/988.0 kB 8.8 MB/s  0:00:00

   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   ------------

In [23]:
import os
from dotenv import load_dotenv

In [30]:
def warn(*args, **kwargs):
    pass

import warnings
warnings.warn = warn
warnings.filterwarnings("ignore")

import wget

# LangChain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

# IBM watsonx
from ibm_watsonx_ai.foundation_models import Model
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import (
    ModelTypes,
    DecodingMethods,
)
from langchain_ibm import WatsonxLLM

from langchain_openai import ChatOpenAI

print("All imports successful!")

All imports successful!


### Import libraries

In [31]:
def warn(*args, **kwargs):
    pass

import warnings
warnings.warn = warn
warnings.filterwarnings("ignore")

import wget

# LangChain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

# IBM watsonx
from ibm_watsonx_ai.foundation_models import Model
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import (
    ModelTypes,
    DecodingMethods,
)
from langchain_ibm import WatsonxLLM

from langchain_openai import ChatOpenAI

print("All imports successful!")

All imports successful!


### Preprocessing - Load the document

In [4]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Use wget to download the file
wget.download(url, out=filename)
print('file downloaded')

file downloaded


##### View Document

In [5]:
with open(filename, 'r') as file:
    # Read the contents of the file
    contents = file.read()
    print(contents)

1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potential violations of 

##### Splitting the document into Chunks

In [8]:
loader = TextLoader(filename)
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
text = text_splitter.split_documents(documents)
print(len(text))

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16


##### Embedding and storing

In [12]:
embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(text, embeddings)
print('Document Ingested')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 663.47it/s]


Document Ingested


##### LLM Model Construction

In this section, we build an LLM model

In [57]:
model_id = 'nvidia/nemotron-3.5-super-vl:free'

In [58]:
load_dotenv()

api_key = os.getenv('OPENROUTER_API_KEY')

llm = ChatOpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=api_key,
    model=model_id,
    temperature=0.5
)

##### Integrating LangChain

In [59]:
qa = RetrievalQA.from_chain_type(
    llm = llm,
    chain_type='stuff',
    retriever=docsearch.as_retriever(),
    return_source_documents=False)

query = 'What is mobile policy?'
response = qa.invoke({"query":query})
print(response['result'])

Based on the provided context, the **Mobile Phone Policy** is a set of standards and expectations that govern the appropriate and responsible usage of mobile devices within the organization.

Its purpose is to ensure employees use mobile phones in a manner consistent with company values and legal compliance. Key areas covered by the policy include:

*   **Acceptable Use:** Mobile devices are primarily for work-related tasks; limited personal use is allowed if it does not disrupt work.
*   **Security:** Safeguarding devices and credentials, being cautious with apps and links, and reporting suspicious activity.
*   **Confidentiality:** Avoiding transmission of sensitive company information via unsecured messaging or email, and being discreet in public.
*   **Cost Management:** Keeping personal usage separate from company accounts and reimbursing personal charges on company-issued phones.
*   **Compliance:** Adhering to relevant laws and regulations, including data protection and privacy.

In [63]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=docsearch.as_retriever(),
    return_source_documents=False                             
                                 )
query = 'Can you summarize the document for me?'
response = qa.invoke({"query":query})
print(response['result'])

Based on the provided text, here is a summary of the document:

**Code of Conduct**
The document outlines a Code of Conduct built on five core principles:
*   **Integrity:** Acting honestly and transparently, protecting sensitive information, and avoiding conflicts of interest.
*   **Respect:** Embracing diversity, prohibiting discrimination and harassment, and fostering an inclusive environment.
*   **Accountability:** Taking responsibility for actions, complying with laws and regulations, and reporting potential violations.
*   **Safety:** Prioritizing the safety of employees, clients, and communities, and reporting unsafe conditions.
*   **Environmental Responsibility:** Minimizing environmental impact and promoting sustainability.

**Health and Safety Policy**
This section emphasizes a paramount commitment to the well-being of employees, customers, and the public. Key commitments include complying with health and safety laws, maintaining a hazard-free workplace, preventing accident

##### Dive Deeper

In [67]:
# As you can see, the query is asking something that does not exist in the document. 
# The LLM responds with information that actually is not true. 
# You don't want this to happen, so you must add a prompt to the LLM.

qa = RetrievalQA.from_chain_type(llm=llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "Can I eat in company vehicles?"
response = qa.invoke({"query": query})
print(response['result'])

I don't know. The provided policies mention that smoking is not permitted in company vehicles, but none of them specifically address whether eating is allowed.


##### Using prompt template

In [69]:
promtp_template = """"Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definitely do not try to make up an answer.
{context}
Question: {question}
"""

PROMPT = PromptTemplate(template=promtp_template, input_variables=['context', 'question'])

chain_type_kwargs = {"prompt": PROMPT}

In [71]:
qa = RetrievalQA.from_chain_type(
    llm,
    chain_type = "stuff",
    retriever=docsearch.as_retriever(),
    chain_type_kwargs=chain_type_kwargs,
    return_source_documents=False
)

query = 'Can I eat in company vehicles?'
response = qa.invoke({"query": query})
print(response['result'])

I don't know. The document does not mention whether eating is permitted in company vehicles. It only states that smoking is not permitted in company vehicles.


In [72]:
query = "What I cannot do in it?"
qa.invoke(query)

{'query': 'What I cannot do in it?',
 'result': 'Based on the **Internet and Email Policy** and the **Mobile Phone Policy**, here are the things you cannot do:\n\n### Internet and Email Policy\n- Use company internet and email services primarily for non-job-related tasks (limited personal use is only allowed during non-work hours and must not interfere with work).\n- Share your login credentials or passwords with others.\n- Open or click on email attachments and links from unknown sources without exercising caution.\n- Send confidential information, trade secrets, or sensitive customer data via email unless encryption is applied.\n- Discuss company matters carelessly on public forums or social media.\n- Use the internet or email for harassment, discrimination, or to distribute offensive or inappropriate content.\n- Violate any relevant laws or regulations, including those related to copyright and data protection.\n\n### Mobile Phone Policy\n- Use your mobile device primarily for non-wo

##### Make the conversation have memory

In [74]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [77]:
qa = ConversationalRetrievalChain.from_llm(
    llm = llm,
    chain_type = 'stuff',
    retriever=docsearch.as_retriever(),
    memory = memory,
    get_chat_history=lambda h : h,
    return_source_documents = False
)

history = []

In [80]:
query = 'What is mobile policy?'
response = qa.invoke({'question': query}, {'chat_history': history})
print(response['answer'])

Based on the provided context, the **Mobile Phone Policy** is the organization's policy that sets forth the standards and expectations for the appropriate and responsible usage of mobile devices in the workplace.

Its purpose is to ensure employees use mobile phones in a manner consistent with company values and legal compliance. Key areas covered include:

*   **Acceptable Use:** Mobile devices are primarily for work-related tasks; limited personal use is allowed if it does not disrupt work obligations.
*   **Security:** Employees must safeguard their devices and access credentials, use caution when downloading apps or clicking links from unfamiliar sources, and promptly report security concerns.
*   **Confidentiality:** Sensitive company information should not be transmitted via unsecured messaging apps or emails, and employees should be discreet when discussing company matters in public.
*   **Cost Management:** Personal phone usage should be kept separate from company accounts, and

In [81]:
history.append((query, response['answer']))

In [82]:
query = 'List points in it?'
response = qa({'question': query}, {'chat_history': history})
print(response['answer'])

Based on the Mobile Phone Policy in the context, the points are:

1. **Acceptable Use** – Mobile devices are primarily for work-related tasks; limited personal usage is allowed if it doesn't disrupt work obligations.
2. **Security** – Safeguard your device and access credentials; exercise caution with unfamiliar apps or links; report security concerns promptly.
3. **Confidentiality** – Avoid transmitting sensitive company information via unsecured messaging apps or emails; be discreet when discussing company matters in public.
4. **Cost Management** – Keep personal phone usage separate from company accounts; reimburse the company for personal charges on company-issued phones.
5. **Compliance** – Adhere to all pertinent laws and regulations concerning mobile phone usage, including data protection and privacy.
6. **Lost or Stolen Devices** – Immediately report any lost or stolen mobile devices to the IT department or your supervisor.
7. **Consequences** – Non-compliance may lead to disci

In [83]:
history.append((query, response['answer']))

In [84]:
query = 'What is the aim of it?'
response = qa({'question': query}, {'chat_history': history})
print(response['answer'])

The Mobile Phone Policy is aimed at promoting the responsible and secure use of mobile devices in line with legal and ethical standards.


##### Wrap up and Make it an Agent

In [88]:
def qa():
        memory = ConversationBufferMemory(memory_key="chat_history", return_message = True)
        qa = ConversationalRetrievalChain.from_llm(
                llm=llm,
                chain_type='stuff',
                retriever=docsearch.as_retriever(),
                memory = memory,
                get_chat_history=lambda h : h,
                return_source_documents=False)

        history=[]

        while True:
                query = input('Question: ')

                if query.lower() in['quit', 'exit', 'bye']:
                        print('Answer: Goodbye!')
                        break
                response = qa({'question': query}, {'chat_history': history})
                history.append((query, response['answer']))
                print('Answer: ', response['answer'])


In [89]:
qa()

Answer:  Based on the context provided, the **Health and Safety Policy** appears to be the most important topic, as the text states: "Our commitment to health and safety is **paramount**."
Answer:  Based on the provided context, the policy for working with colleagues is outlined in the **Code of Conduct**. Key principles include:

*   **Integrity:** Act honestly and transparently in all interactions with colleagues.
*   **Respect:** Embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. Everyone should be treated with dignity and courtesy in an inclusive environment where differences are celebrated.
*   **Accountability:** Take responsibility for your actions and decisions, follow relevant laws and regulations, and report any potential violations of the code.

Additionally, the **Health and Safety Policy** and **Anti-discrimination and Harassment Policy** reinforce the commitment to a safe, healthy